In [ ]:
from datetime import datetime
from opera_utils import disp, tropo
import geopandas as gpd
import os
import sys
from pathlib import Path
import subprocess
from transboundary_opera import displacement_tools as dt
import matplotlib.pyplot as plt
import xarray as xr
from shapely.geometry import mapping
import ultraplot as uplt
from pyproj import CRS
import asf_search as asf
import re
import numpy as np
import rasterio as rio
from opera_utils.download import L2Product
from transboundary_opera import run1_download_DISP_S1_Static, run2_prep_mintpy_opera
from opera_utils.disp import _download, _reformat, mintpy

# %matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# Read username (line 1) and password (line 2) from a text file and set env vars
cred_file = Path("/home/clayc/earthdata_info.txt")  # update this path

with cred_file.open("r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines()]

if len(lines) < 2 or not lines[0] or not lines[1]:
    raise ValueError("Credential file must have username on line 1 and password on line 2.")

username, password = lines[0], lines[1]

os.environ["EARTHDATA_USERNAME"] = username
os.environ["EARTHDATA_PASSWORD"] = password

print("EARTHDATA credentials loaded and environment variables set.")

In [ ]:
SEARCH_START = datetime(2024, 6, 1)     # next runs should start at Jan 1, 2016
SEARCH_END = datetime(2025, 1, 1)

gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")
# frame_ids = dt.get_unique_frame_ids(gdf)     # this only gets one set of frame ids 
# display(frame_ids)

In [ ]:
unique_frame_ids = [1092,1093,1094,1095,1096,1097,3065,3066,3067,5124,5125,7091,7092,7093,14877,14878,14879,16939,16940,18905,20694,20695,20696,20697,22665,22666,22667,24724,24725,26691,26692,28478,28479,28480,28481,28482,30452,30453,30454,30455,34477,34478,34479,38241,40295,40296,40297,40298,42264,42265,44325,44326,46290,46291]

In [ ]:
def prep_OPERA_disp_for_mintpy(
        tgt_frame,
        out_dir,
        start_datetime,
        end_datetime
):
    
    out_path = out_dir / 'OPERA_data' / str(tgt_frame)

    if (out_path / 'subset-ncs').exists() == False:
        _download.run_download(
                        tgt_frame,
                        start_datetime= start_datetime,
                        end_datetime = end_datetime,
                        num_workers = 8,
                        output_dir = out_path / 'subset-ncs'   # TODO: SHOULD CHANGE THIS TO 'subset-ncs'
                        
                        # bbox: tuple[float, float, float, float] | None = None,    # leaving these out to get entire frame, 
                        # wkt: str | None = None,                                   # will get zonal statistics later with boundaries
        )
    else:
        print(f'OPERA DISP .nc files already downloaded for {tgt_frame}')

    staticDir = out_path / 'orbit_data'
    os.makedirs(staticDir, exist_ok=True)
    dispDir = out_path / 'disp_data'
    os.makedirs(dispDir, exist_ok=True)
    geomDir = out_path / 'geom_data'
    os.makedirs(geomDir, exist_ok=True)

    sys.argv = [
        'notebook',  # placeholder for script name
        '--frameID', str(tgt_frame),
        # '--version', 0.9,
        '--startDate', start_datetime.strftime('%Y%m%d'),
        '--endDate', end_datetime.strftime('%Y%m%d'),
        '--dispDir', str(dispDir),
        '--staticDir', str(staticDir),
        '--geomDir', str(geomDir),
        '--staticOnly'                             # default is true, uncomment for false
    ]

    run1_inps = run1_download_DISP_S1_Static.createParser()
    run1_download_DISP_S1_Static.main(run1_inps)

    timeseries_path = geomDir.parent / 'mintpy' / 'timeseries.h5'
    
    # Check both directory AND file existence
    if timeseries_path.exists():
        try:
            # TODO: may need to apply a mask to ensure same dimensions between geometry and deformation data

            losEast = rio.open(geomDir / 'los_east.tif').read(1)
            losNorth = rio.open(geomDir / 'los_north.tif').read(1)
            up = np.sqrt(1 - losEast**2 - losNorth**2)

            with rio.open(geomDir / 'los_enu.tif', 'w',
                         driver='GTiff',
                         height=losEast.shape[0],
                         width=losEast.shape[1],
                         count=3,
                         dtype=losEast.dtype) as dst:
                dst.write(losEast, 1)
                dst.write(losNorth, 2)
                dst.write(up, 3)
                
        except BlockingIOError:
                print(f"Frame {tgt_frame}: timeseries.h5 still locked, skipping for now")
                return

    os.rmdir(dispDir)   # DISP products are already downloaded 

    nc_files = list(out_path.glob('*/*.nc'))
    geomDir = out_path / 'geom_data'

    _reformat.reformat_stack(
        input_files = nc_files,
        output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
        apply_ionospheric_corrections = True,

        # drop_vars: Sequence[str] | None = None,
        reference_method = 'border',    # 'high_coherence' or 'border' Border probably better since we may mosaic for decomposition?
        # reference_border_pixels: int = 3,
        reference_coherence_threshold = 0.7
    )

    mintpy.disp_nc_to_mintpy(
        str(out_path / f'disp-stack-{out_path.stem}.nc'),
        sample_disp_nc = nc_files[0],
        los_enu_path = geomDir / 'los_enu.tif',
        dem_path = None,
        layover_shadow_mask_path = geomDir / 'layover_shadow_mask.tif',
        outdir = out_path / "mintpy",
        process_chunk_size = (512, 512),
    )

In [ ]:
for frame in unique_frame_ids[:5]:
    prep_OPERA_disp_for_mintpy(
        frame,
        Path('/mnt/c/Users/clayc/Documents/US_Mex_InSAR'),  # wsl; # Path('/home/clayc/Documents/US_Mex_InSAR'),     # linux
        SEARCH_START,
        SEARCH_END
        )

## Step 1. Download the OPERA DISP NetCDF files

In [ ]:
tgt_frame = unique_frame_ids[2]

out_path = Path('/mnt/c/Users/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(tgt_frame)
os.makedirs(out_path, exist_ok=True)
display(out_path)

nc_files = list(out_path.glob('*/*.nc'))
geomDir = out_path / 'geom_data'

In [ ]:
_download.run_download(
                        tgt_frame,
                        start_datetime= SEARCH_START,
                        end_datetime = SEARCH_END,
                        num_workers = 8,
                        output_dir = out_path / 'subset-ncs'   # TODO: SHOULD CHANGE THIS TO 'subset-ncs'
                        
                        # bbox: tuple[float, float, float, float] | None = None,    # leaving these out to get entire frame, 
                        # wkt: str | None = None,                                   # will get zonal statistics later with boundaries
    
)

## Step 2. Download OPERA Static layers for viewing geometry
- Also create los_enu.tif for future decomposition

In [ ]:
staticDir = out_path / 'orbit_data'
os.makedirs(staticDir, exist_ok=True)
dispDir = out_path / 'disp_data'
os.makedirs(dispDir, exist_ok=True)
geomDir = out_path / 'geom_data'
os.makedirs(geomDir, exist_ok=True)

sys.argv = [
    'notebook',  # placeholder for script name
    '--frameID', str(tgt_frame),
    # '--version', 0.9,
    '--startDate', SEARCH_START.strftime('%Y%m%d'),
    '--endDate', SEARCH_END.strftime('%Y%m%d'),
    '--dispDir', str(dispDir),
    '--staticDir', str(staticDir),
    '--geomDir', str(geomDir),
    '--staticOnly'                             # default is true, uncomment for false
]

run1_inps = run1_download_DISP_S1_Static.createParser()
run1_download_DISP_S1_Static.main(run1_inps)

timeseries_path = geomDir.parent / 'mintpy' / 'timeseries.h5'
    
# Check both directory AND file existence
if timeseries_path.exists():
    try:
        # TODO: may need to apply a mask to ensure same dimensions between geometry and deformation data
        losEast = rio.open(geomDir / 'los_east.tif').read(1)
        losNorth = rio.open(geomDir / 'los_north.tif').read(1)
        up = np.sqrt(1 - losEast**2 - losNorth**2)
        with rio.open(geomDir / 'los_enu.tif', 'w',
                     driver='GTiff',
                     height=losEast.shape[0],
                     width=losEast.shape[1],
                     count=3,
                     dtype=losEast.dtype) as dst:
            dst.write(losEast, 1)
            dst.write(losNorth, 2)
            dst.write(up, 3)
            
    except BlockingIOError:
        print(f"Frame {tgt_frame}: timeseries.h5 still locked, skipping for now")
        # continue

os.rmdir(dispDir)   # DISP products are already downloaded 

In [ ]:
for frame in unique_frame_ids:
    staticDir = out_path / 'orbit_data'
    os.makedirs(staticDir, exist_ok=True)
    dispDir = out_path / 'disp_data'
    os.makedirs(dispDir, exist_ok=True)
    geomDir = out_path / 'geom_data'
    os.makedirs(geomDir, exist_ok=True)

    sys.argv = [
        'notebook',  # placeholder for script name
        '--frameID', str(frame),
        # '--version', 0.9,
        '--startDate', SEARCH_START.strftime('%Y%m%d'),
        '--endDate', SEARCH_END.strftime('%Y%m%d'),
        '--dispDir', str(dispDir),
        '--staticDir', str(staticDir),
        '--geomDir', str(geomDir),
        '--staticOnly'                             # default is true, uncomment for false
    ]

    run1_inps = run1_download_DISP_S1_Static.createParser()
    run1_download_DISP_S1_Static.main(run1_inps)

    timeseries_path = geomDir.parent / 'mintpy' / 'timeseries.h5'
    
    # Check both directory AND file existence
    if timeseries_path.exists():
        try:
            # TODO: may need to apply a mask to ensure same dimensions between geometry and deformation data

            losEast = rio.open(geomDir / 'los_east.tif').read(1)
            losNorth = rio.open(geomDir / 'los_north.tif').read(1)
            up = np.sqrt(1 - losEast**2 - losNorth**2)

            with rio.open(geomDir / 'los_enu.tif', 'w',
                         driver='GTiff',
                         height=losEast.shape[0],
                         width=losEast.shape[1],
                         count=3,
                         dtype=losEast.dtype) as dst:
                dst.write(losEast, 1)
                dst.write(losNorth, 2)
                dst.write(up, 3)
                
        except BlockingIOError:
            print(f"Frame {frame}: timeseries.h5 still locked, skipping for now")
            continue

    os.rmdir(dispDir)   # DISP products are already downloaded 

In [ ]:
losEast = rio.open(geomDir / 'los_east.tif').read(1)
losNorth = rio.open(geomDir / 'los_north.tif').read(1)
up = np.sqrt(1 - losEast**2 - losNorth**2)

with rio.open(geomDir / 'los_enu.tif', 'w',
            driver='GTiff',
            height=losEast.shape[0],
            width=losEast.shape[1],
            count=3,
            dtype=losEast.dtype) as dst:
        dst.write(losEast, 1)
        dst.write(losNorth, 2)
        dst.write(up, 3)

## Step 3. reformat time series stacks for each frame

In [ ]:
test_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(unique_frame_ids[0])   # linux
print(test_path)
nc_files = list(test_path.glob('*/*.nc'))

In [ ]:
from netCDF4 import Dataset
test_file = Dataset(nc_files[0])

temporal_coherence = test_file.variables['temporal_coherence'][:]
max_coherence = np.nanmax(temporal_coherence)
print(f"Maximum temporal coherence: {max_coherence}")

y_idx, x_idx = np.unravel_index(np.nanargmax(temporal_coherence), temporal_coherence.shape)
y_coord = test_file.variables['y'][y_idx]
x_coord = test_file.variables['x'][x_idx]
# print(f"Maximum coherence location: y={y_coord}, x={x_coord} (indices: y_idx={y_idx}, x_idx={x_idx})")

In [ ]:
geomDir = test_path / 'geom_data'

_reformat.reformat_stack(
    input_files = nc_files,
    output_name = str(test_path / f'disp-stack-{test_path.stem}.nc'),
    apply_ionospheric_corrections = True,
    reference_method = "point",
    reference_row=y_idx,
    reference_col=x_idx
)

In [ ]:
for frame in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame)
    os.makedirs(out_path, exist_ok=True)

    nc_files = list(out_path.glob('*/*.nc'))
    geomDir = out_path / 'geom_data'

    if Path(str(out_path / f'disp-stack-{out_path.stem}.nc')).exists() == True:
        continue
    else:
        try:
            test_file = Dataset(nc_files[0])

            temporal_coherence = test_file.variables['temporal_coherence'][:]
            max_coherence = np.nanmax(temporal_coherence)

            y_idx, x_idx = np.unravel_index(np.nanargmax(temporal_coherence), temporal_coherence.shape)
            y_coord = test_file.variables['y'][y_idx]
            x_coord = test_file.variables['x'][x_idx]

            _reformat.reformat_stack(
                input_files = nc_files,
                output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
                apply_ionospheric_corrections = True,
                reference_method = 'point',    
                reference_row=y_idx,
                reference_col=x_idx
            )
        except Exception as e:
            print(f"Frame {frame} failed: {e}")
            continue

In [ ]:
from netCDF4 import Dataset

for frame in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame)
    os.makedirs(out_path, exist_ok=True)

    nc_files = list(out_path.glob('*/*.nc'))
    geomDir = out_path / 'geom_data'

    try:
        test_file = Dataset(nc_files[0])

        temporal_coherence = test_file.variables['temporal_coherence'][:]
        max_coherence = np.nanmax(temporal_coherence)

        y_idx, x_idx = np.unravel_index(np.nanargmax(temporal_coherence), temporal_coherence.shape)
        y_coord = test_file.variables['y'][y_idx]
        x_coord = test_file.variables['x'][x_idx]

        _reformat.reformat_stack(
            input_files = nc_files,
            output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
            apply_ionospheric_corrections = True,
            reference_method = 'point',    
            reference_row=y_idx,
            reference_col=x_idx
        )
    except Exception as e:
        print(f"Frame {frame} failed: {e}")
        continue

## Step 4. prepare OPERA DISP data into a MintPy compatible format

In [ ]:
for frame in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame)
    nc_files = list(out_path.glob('*/*.nc'))
    geomDir = out_path / 'geom_data'

    try:
        mintpy.disp_nc_to_mintpy(
            str(out_path / f'disp-stack-{out_path.stem}.nc'),
            sample_disp_nc = nc_files[0],
            los_enu_path = geomDir / 'los_enu.tif',
            dem_path = None,
            layover_shadow_mask_path = geomDir / 'layover_shadow_mask.tif',
            outdir = out_path / "mintpy"    
            # process_chunk_size = (512, 512)
        )
    except Exception as e:
        print(f"Frame {frame} failed: {e}")
        continue

In [ ]:
mintpy.disp_nc_to_mintpy(
        str(test_path / f'disp-stack-{test_path.stem}.nc'),
        sample_disp_nc = nc_files[0],
        los_enu_path = geomDir / 'los_enu.tif',
        dem_path = None,
        layover_shadow_mask_path = geomDir / 'layover_shadow_mask.tif',
        outdir = test_path / "mintpy"    
        # process_chunk_size = (512, 512)
    )

In [ ]:
import h5py
testgeom = h5py.File(Path('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy') / 'geometryGeo.h5', 'r')
list(testgeom.keys())

In [ ]:
from pyproj import Transformer
import h5py
import numpy as np

# Convert to lat/lon
transformer = Transformer.from_crs(f"EPSG:{dict(testgeom.attrs)['EPSG']}", "EPSG:4326", always_xy=True)
ref_lon, ref_lat = transformer.transform(x_coord, y_coord)

# Convert to native Python types (not numpy)
ref_lat = float(ref_lat)
ref_lon = float(ref_lon)
ref_y = int(y_idx)
ref_x = int(x_idx)

# Add to timeseries.h5
timeseries_file = '/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/timeseries.h5'
with h5py.File(timeseries_file, 'r+') as f:
    f.attrs['REF_LAT'] = ref_lat
    f.attrs['REF_LON'] = ref_lon
    f.attrs['REF_Y'] = ref_y
    f.attrs['REF_X'] = ref_x
    print(f"Added to {timeseries_file}")
    print(f"  REF_LAT: {ref_lat:.6f}")
    print(f"  REF_LON: {ref_lon:.6f}")
    print(f"  REF_Y: {ref_y}")
    print(f"  REF_X: {ref_x}")

# Add to geometryGeo.h5
geometry_file = '/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/geometryGeo.h5'
with h5py.File(geometry_file, 'r+') as f:
    f.attrs['REF_LAT'] = ref_lat
    f.attrs['REF_LON'] = ref_lon
    f.attrs['REF_Y'] = ref_y
    f.attrs['REF_X'] = ref_x
    print(f"Added to {geometry_file}")

In [ ]:
from pyproj import Transformer
import h5py

# Convert to lat/lon
# Using UTM Zone 14N (EPSG:32614) based on your attributes
transformer = Transformer.from_crs(f"EPSG:{dict(testgeom.attrs)['EPSG']}", "EPSG:4326", always_xy=True)
ref_lon, ref_lat = transformer.transform(x_coord, y_coord)

# Add to timeseries.h5
timeseries_file = '/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/timeseries.h5'
with h5py.File(timeseries_file, 'r+') as f:
    f.attrs['REF_LAT'] = ref_lat
    f.attrs['REF_LON'] = ref_lon
    f.attrs['REF_Y'] = y_idx
    f.attrs['REF_X'] = x_idx
    print(f"Added to {timeseries_file}")
    print(f"  REF_LAT: {ref_lat:.6f}")
    print(f"  REF_LON: {ref_lon:.6f}")
    pass

# Add to geometryGeo.h5
geometry_file = '/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/geometryGeo.h5'
with h5py.File(geometry_file, 'r+') as f:
    f.attrs['REF_LAT'] = ref_lat
    f.attrs['REF_LON'] = ref_lon
    f.attrs['REF_Y'] = y_idx
    f.attrs['REF_X'] = x_idx
    print(f"Added to {geometry_file}")
    pass

# Devlopment work past here

- Geometry will be generated using run1_download_DISP_S1_Static. This function generates los_east.tif and los_north.tif. These tifs will be used to determine the azimuth and incidence angles needed for decomposing the LOS displacement

In [ ]:
out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(unique_frame_ids[0])
os.makedirs(out_path, exist_ok=True)

In [ ]:
_download.run_download(
                        unique_frame_ids[0],
                        start_datetime= SEARCH_START,
                        end_datetime = SEARCH_END,
                        num_workers = 8,
                        output_dir = out_path
                        # bbox: tuple[float, float, float, float] | None = None,    # leaving these out to get entire frame, 
                        # wkt: str | None = None,                                   # will get zonal statistics later with boundaries
    
)


In [ ]:
nc_files = list(out_path.glob('*/*.nc'))
geomDir = out_path / 'geom_data'

In [ ]:
_reformat.reformat_stack(
        input_files = nc_files,
        output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
        apply_ionospheric_corrections = True,

        # drop_vars: Sequence[str] | None = None,
        reference_method = 'high_coherence',    # order 'border'
        # reference_border_pixels: int = 3,
        reference_coherence_threshold = 0.7
)

In [ ]:
geomDir = out_path / 'geom_data'

# create appropriate LNU tif for los_enu_path.tif
# probably need to clip to the same extent as the nc
losEast = rio.open(geomDir / 'los_east.tif').read(1)
losNorth = rio.open(geomDir / 'los_north.tif').read(1)
up = np.sqrt(1 - losEast**2 - losNorth**2)

with rio.open(geomDir / 'los_enu.tif', 'w',
            driver='GTiff',
            height=losEast.shape[0],
            width=losEast.shape[1],
            count=3,
            dtype=losEast.dtype) as dst:
        dst.write(losEast, 1)
        dst.write(losNorth, 2)
        dst.write(up, 3)


In [ ]:
mintpy.disp_nc_to_mintpy(
        str(out_path / f'disp-stack-{out_path.stem}.nc'),
        sample_disp_nc = nc_files[0],
        los_enu_path = geomDir / 'los_enu.tif',
        dem_path = None,
        layover_shadow_mask_path = geomDir / 'layover_shadow_mask.tif',
        outdir = out_path / "mintpy",
    )

In [ ]:
import h5py
testgeom = h5py.File(Path('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy') / 'geometryGeo.h5', 'r')

In [ ]:
from opera_utils.disp import _download

for frame_id in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame_id)
    os.makedirs(out_path, exist_ok=True)


    _download.run_download(
                        frame_id,
                        start_datetime= SEARCH_START.strftime("%Y-%m-%d"),
                        end_datetime = SEARCH_END.strftime("%Y-%m-%d"),
                        num_workers = 8,
                        output_dir = out_path
                        # bbox: tuple[float, float, float, float] | None = None,    # leaving these out to get entire frame, 
                        # wkt: str | None = None,                                   # will get zonal statistics later with boundaries
    
)

In [ ]:
# path to mintpy bash script
script_path = Path(os.getcwd()).parent / 'create-mintpy_cc.sh'

for frame_id in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame_id)
    os.makedirs(out_path, exist_ok=True)

    if (out_path / "mintpy").exists():
        print(f"Frame ID {frame_id} already processed. Skipping.")
        continue
    
    else:
        # 2. Define the REQUIRED environment variables for the script
        required_env = {
            "FRAME_ID": f"{frame_id}",  # Using the current frame ID from the loop
            "START": SEARCH_START.strftime("%Y-%m-%d"),
            "END": SEARCH_END.strftime("%Y-%m-%d"),
            }

        # 3. Define OPTIONAL environment variables to override the script's defaults
        optional_env = {
            "NUM_WORKERS": "8",              # Overrides default of 4
            "REFERENCE_METHOD": "HIGH_COHERENCE"    # Overrides default of BORDER. Either HIGH_COHERENCE or BORDER would be best I think.
        }
        print(f"Preparing to execute script: {script_path}")
        print(f"With required environment: {required_env}")

        # Construct the full environment: current system environment + custom variables
        full_environment = os.environ.copy()
        full_environment.update(required_env)
        full_environment.update(optional_env)

        try:
            # Execute the script directly (preferred over shell=True)
            result = subprocess.run(
                [script_path],
                check=True,  # Raise CalledProcessError for non-zero exit codes
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,  # Decode output as string
                env=full_environment,
                cwd = out_path
            )

            # Success output
            print("\nScript Execution Successful")
            print(f"Return Code: {result.returncode}")
            print("\nSTDOUT")
            print(result.stdout)
            if result.stderr:
                print("\nSTDERR (Non-fatal messages)")
                print(result.stderr)
        except subprocess.CalledProcessError as e:
            # Error handling for script failure (e.g., if a command inside the script fails)
            print(f"\nERROR: Script Failed (Exit Code {e.returncode})")
            print(f"Command run: {e.cmd}")
            print("\nSTDOUT (Captured before failure)")
            print(e.stdout)
            print("\nSTDERR (Error messages)")
            print(e.stderr)

        except FileNotFoundError:
            # Error handling for when the script file itself cannot be found
            print(f"\nCRITICAL ERROR: Script file not found or lacks execute permission at: {script_path}")

### Use run1_download_DISP_S1_Static to download CLSC Static files needed for orbit geometry

In [ ]:
import h5py

def clip_geometry_to_displacement(geom_file, ts_file):
    """
    Clip geometry arrays in memory to match displacement spatial extent
    
    Returns:
        dict: Dictionary of clipped geometry arrays (inc_angle, azi_angle, etc.)
        dict: Attributes from timeseries file
    """
    with h5py.File(ts_file, 'r') as f:
        ts_atr = dict(f.attrs)
        ts_shape = f['timeseries'].shape[1:]  # (length, width)
    
    with h5py.File(geom_file, 'r') as f:
        geom_atr = dict(f.attrs)
        
        # Calculate pixel offsets
        ts_lat0 = float(ts_atr['Y_FIRST'])
        ts_lon0 = float(ts_atr['X_FIRST'])
        ts_lat_step = float(ts_atr['Y_STEP'])
        ts_lon_step = float(ts_atr['X_STEP'])
        
        geom_lat0 = float(geom_atr['Y_FIRST'])
        geom_lon0 = float(geom_atr['X_FIRST'])
        geom_lat_step = float(geom_atr['Y_STEP'])
        geom_lon_step = float(geom_atr['X_STEP'])
        
        # Calculate starting indices in geometry file
        y0 = int(round((ts_lat0 - geom_lat0) / geom_lat_step))
        x0 = int(round((ts_lon0 - geom_lon0) / geom_lon_step))
        
        # Clip all geometry datasets
        clipped_geom = {}
        for key in f.keys():
            clipped_geom[key] = f[key][y0:y0+ts_shape[0], x0:x0+ts_shape[1]]
    
    return clipped_geom, ts_atr

In [ ]:
for frame in unique_frame_ids:
    staticDir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame) / 'orbit_data'
    os.makedirs(staticDir, exist_ok=True)
    dispDir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame) / 'disp_data'
    os.makedirs(dispDir, exist_ok=True)
    geomDir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame) / 'geom_data'
    os.makedirs(geomDir, exist_ok=True)

    sys.argv = [
        'notebook',  # placeholder for script name
        '--frameID', str(frame),
        # '--version', 0.9,
        '--startDate', SEARCH_START.strftime('%Y%m%d'),
        '--endDate', SEARCH_END.setrftime('%Y%m%d'),
        '--dispDir', str(dispDir),
        '--staticDir', str(staticDir),
        '--geomDir', str(geomDir),
        '--staticOnly'                             # default is true, uncomment for false
    ]

    run1_inps = run1_download_DISP_S1_Static.createParser()
    run1_download_DISP_S1_Static.main(run1_inps)

    timeseries_path = geomDir.parent / 'mintpy' / 'timeseries.h5'
    
    # Check both directory AND file existence
    if timeseries_path.exists():
        try:
            losEast = rio.open(geomDir / 'los_east.tif').read(1)
            losNorth = rio.open(geomDir / 'los_north.tif').read(1)
            up = np.sqrt(1 - losEast**2 - losNorth**2)

            # dt.create_geom_h5_with_ref(
            #     geomDir / 'los_east.tif', 
            #     geomDir / 'los_north.tif', 
            #     timeseries_path
            # )

            with rio.open(geomDir / 'los_enu.tif', 'w',
                         driver='GTiff',
                         height=losEast.shape[0],
                         width=losEast.shape[1],
                         count=3,
                         dtype=losEast.dtype) as dst:
                dst.write(losEast, 1)
                dst.write(losNorth, 2)
                dst.write(up, 3)
                
        except BlockingIOError:
            print(f"Frame {frame}: timeseries.h5 still locked, skipping for now")
            continue

## Trying out a different approach for downloading and preparing the data:
1. run /src/transboundary_opera/run1_download_DISP_S1_Static.py to download the OPERA DISP (displacement time series) and CSLC (orbit geometry) products
2. run /src/transboundary_opera/run2_prep_mintpy_opera.py

In [ ]:
# Args:
# --frameID    OPERA frame number
# --version    OPERA dataset version
# --staticDir  Folder for static layers/metadata
# --geomDir    Folder for geometry files
# --dispDir    Folder for data
# --startDate  Start date (optional)
# --endDate    End date (optional)

run1_download_DISP_S1_Static.py \
      --frameID 11116 \
      --version 0.9 \
      --staticDir /path/to/work/folder/static_lyrs \
      --geomDir /path/to/work/folder/geometry \
      --dispDir /path/to/work/folder/data #\
     #--startDate 20170101
     #--endDate 20190101

In [ ]:
## Example Command to Run `run2_prep_mintpy_opera.py`
# Args:
# -m   Folder for static layers/metadata
# -u   Folder with data (*.nc for all files)
# -g   Folder for geometry files
# -o   Folder for timeseries output
# --water-mask-file  Water mask file (auto-generated)
# --dem-file         DEM file (auto-generated)
# --ref-lalo         Spatial reference for timeseries
# --apply-mask       Apply mask (optional)

run2_prep_mintpy_opera.py \
        -m "/path/to/work/folder/static_lyrs" \
        -u "/path/to/work/folder/data/*.nc" \
        -g "/path/to/work/folder/geometry" \
        -o /path/to/work/folder/mintpy_output \
        --water-mask-file esa_world_cover_2021 \
        --dem-file glo_30 \
        --ref-lalo '36.612 -121.064' \
        --apply-mask

In [ ]:
sys.argv = [
    "notebook", # placeholder for script name
    "--unw-file-glob",
    "--geom-dir",
    "--meta-file", 
    "--start-date", SEARCH_START.strftime('%Y%m%d'),    # start of search window for OPERA products; any ifgs earlier will be removed
    "--end-date", SEARCH_END.strftime('%Y%m%d'),        # end of search window for OPERA products; any ifgs later will be removed
    "--out-dir",
    "--range", 1,
    "--azimuth", 1, 
    "--water-mask-file", 
    "--dem-file", 'glo_30',                             # can be path to valid DEM or download one of ssrtm_v3, nasadem, glo_30, glo_90, glo_90_missing
    "--ref-lalo", None,                 # 'latitude longitude' of reference point, or None to select highest coherence point by default
    "--zero-mask", True,                # masks all pixels with zero in the unwrapped phase
    "--corr-lyrs", True,                # extract correction layers
    "--shortwvl-lyrs", True,            # extract short wavelength layers
    "--tropo-correction", True,         # apply tropospheric correction using HRRR weather model
    "--work-dir",                       # Working directory for tropospheric correction intermediates
    "--mask-layers", True,              # extract all mask layers
    "--load-all-layers", True,          # extract all layers
    "--apply-mask", True,               # apply epoch based masking
    "--dem-error", True,                # apply dem error correction
    "--n-workers", 6,                   # default is none; number of workers for writing timeseries in parallel
    "--reliability-threshold", 0.9,     # default of 0.9; 
    "--chunk-size", 50,                 # default of 50
]

run2_inps = run2_prep_mintpy_opera.createParser()
run2_prep_mintpy_opera.main(run2_inps)

In [ ]:
import rasterio as rio

test_east = rio.open(test_geomDir / 'los_east.tif')
test_north = rio.open(test_geomDir / 'los_north.tif')
test_layover = rio.open(test_geomDir / 'layover_shadow_mask.tif')

In [ ]:
los_east = test_east.read(1).flatten()
los_north = test_north.read(1).flatten()

In [ ]:
los_east.flatten()
los_north.flatten()

In [ ]:
plt.imshow(test_layover.read(1))

In [ ]:
run1_download_DISP_S1_Static(frameID = te)

In [ ]:
static_products = {}
product = L2Product.CSLC_STATIC

for frame in unique_frame_ids:
    static_products[frame] = asf.search(
        frame=frame,
        # dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.CSLC_STATIC,
    )

In [ ]:
static_products

In [ ]:
# search CLSC Static Layer files
product = L2Product.CSLC_STATIC

results = asf.search(
    frame=test_frame,
    processingLevel=product.value,
)

# results.download(path=test_staticDir, processes=5)    # downloading static layers with simultaneous downloads